# Challenge 001 — Diagnóstico de Churn
## Carregamento e visualização dos 5 datasets

Datasets: RavenStack (SaaS Subscription & Churn Analytics). Dados em `data/`.

In [98]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

### 1. ravenstack_accounts.csv
Contas: indústria, país, canal de aquisição, plano, flag de trial. Chave: `account_id`.

In [99]:
accounts = pd.read_csv(DATA_DIR / "ravenstack_accounts.csv")
accounts = accounts.rename(columns={
    "account_id": "id_conta",
    "account_name": "nome_conta",
    "industry": "segmento",
    "country": "pais",
    "signup_date": "data_cadastro",
    "referral_source": "canal_aquisicao",
    "plan_tier": "plano",
    "seats": "numero_usuarios",
    "is_trial": "periodo_teste",
    "churn_flag": "flag_churn",
})

#padronizar data_cadastro padrao brasil 

accounts["data_cadastro"] = pd.to_datetime(accounts["data_cadastro"], errors="coerce").dt.strftime("%d/%m/%Y")


# NaN em colunas de texto/categóricas -> "Não fornecido"
for col in accounts.select_dtypes(include="object").columns:
    accounts[col] = accounts[col].fillna("Não fornecido")

print("Shape:", accounts.shape)
print("\nColunas:", list(accounts.columns))
print("\nTipos:")
display(accounts.dtypes)
accounts.head(10)

Shape: (500, 10)

Colunas: ['id_conta', 'nome_conta', 'segmento', 'pais', 'data_cadastro', 'canal_aquisicao', 'plano', 'numero_usuarios', 'periodo_teste', 'flag_churn']

Tipos:


id_conta           object
nome_conta         object
segmento           object
pais               object
data_cadastro      object
canal_aquisicao    object
plano              object
numero_usuarios     int64
periodo_teste        bool
flag_churn           bool
dtype: object

,id_conta,nome_conta,segmento,pais,data_cadastro,canal_aquisicao,plano,numero_usuarios,periodo_teste,flag_churn
0,A-2e4581,Company_0,EdTech,US,16/10/2024,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,17/08/2023,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,27/08/2024,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,27/08/2023,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,27/10/2024,event,Enterprise,35,False,True
5,A-1b9609,Company_5,EdTech,IN,12/10/2023,ads,Enterprise,4,False,False
6,A-a0ca4e,Company_6,Cybersecurity,US,08/03/2024,ads,Pro,11,False,False
7,A-e5d6ab,Company_7,EdTech,US,15/04/2023,partner,Pro,3,False,False
8,A-7dacce,Company_8,Cybersecurity,CA,10/09/2024,event,Enterprise,12,False,True
9,A-10b8da,Company_9,DevTools,US,08/05/2023,partner,Enterprise,14,False,True


### 2. ravenstack_subscriptions.csv
Assinaturas: MRR, ARR, plano, upgrades/downgrades, billing frequency. Chave: `subscription_id` → `account_id`.

In [100]:
subscriptions = pd.read_csv(DATA_DIR / "ravenstack_subscriptions.csv")
subscriptions = subscriptions.rename(columns={
    "subscription_id": "id_assinatura",
    "account_id": "id_conta",
    "start_date": "data_inicio",
    "end_date": "data_fim",
    "plan_tier": "plano",
    "seats": "numero_usuarios",
    "mrr_amount": "mrr_valor",
    "arr_amount": "arr_valor",
    "is_trial": "periodo_teste",
    "upgrade_flag": "flag_upgrade",
    "downgrade_flag": "flag_downgrade",
    "churn_flag": "flag_churn",
    "billing_frequency": "frequencia_cobranca",
    "auto_renew_flag": "flag_renovacao_automatica",
})

# Datas em formato padrão Brasil (dd/mm/yyyy)
subscriptions["data_inicio"] = pd.to_datetime(subscriptions["data_inicio"], errors="coerce").dt.strftime("%d/%m/%Y")
subscriptions["data_fim"] = pd.to_datetime(subscriptions["data_fim"], errors="coerce").dt.strftime("%d/%m/%Y")

# Valores monetários em float
subscriptions["mrr_valor"] = subscriptions["mrr_valor"].astype(float)
subscriptions["arr_valor"] = subscriptions["arr_valor"].astype(float)

# Frequência de cobrança em português
subscriptions["frequencia_cobranca"] = subscriptions["frequencia_cobranca"].replace({
    "monthly": "mensal",
    "annual": "anual",
})

# NaN em colunas de texto/categóricas -> "Não fornecido"
for col in subscriptions.select_dtypes(include="object").columns:
    subscriptions[col] = subscriptions[col].fillna("Não fornecido")

print("Shape:", subscriptions.shape)
print("\nColunas:", list(subscriptions.columns))
print("\nTipos:")
display(subscriptions.dtypes)
subscriptions.head(10)

Shape: (5000, 14)

Colunas: ['id_assinatura', 'id_conta', 'data_inicio', 'data_fim', 'plano', 'numero_usuarios', 'mrr_valor', 'arr_valor', 'periodo_teste', 'flag_upgrade', 'flag_downgrade', 'flag_churn', 'frequencia_cobranca', 'flag_renovacao_automatica']

Tipos:


id_assinatura                 object
id_conta                      object
data_inicio                   object
data_fim                      object
plano                         object
numero_usuarios                int64
mrr_valor                    float64
arr_valor                    float64
periodo_teste                   bool
flag_upgrade                    bool
flag_downgrade                  bool
flag_churn                      bool
frequencia_cobranca           object
flag_renovacao_automatica       bool
dtype: object

,id_assinatura,id_conta,data_inicio,data_fim,plano,numero_usuarios,mrr_valor,arr_valor,periodo_teste,flag_upgrade,flag_downgrade,flag_churn,frequencia_cobranca,flag_renovacao_automatica
0,S-8cec59,A-3c1a3f,23/12/2023,12/04/2024,Enterprise,14,2786.0,33432.0,False,False,False,True,mensal,True
1,S-0f6f44,A-9b9fe9,11/06/2024,Não fornecido,Pro,17,833.0,9996.0,False,False,False,False,mensal,True
2,S-51c0d1,A-659280,25/11/2024,Não fornecido,Enterprise,62,0.0,0.0,True,True,False,False,anual,False
3,S-f81687,A-e7a1e2,23/11/2024,13/12/2024,Enterprise,5,995.0,11940.0,False,False,False,True,mensal,True
4,S-cff5a2,A-ba6516,10/01/2024,Não fornecido,Enterprise,27,5373.0,64476.0,False,False,False,False,mensal,True
5,S-4b9b13,A-fa2041,13/08/2024,Não fornecido,Pro,15,735.0,8820.0,False,False,False,False,mensal,True
6,S-dceac6,A-417d2f,30/12/2023,Não fornecido,Enterprise,4,796.0,9552.0,False,False,False,False,anual,True
7,S-8cad7b,A-5f2961,23/12/2024,Não fornecido,Basic,16,304.0,3648.0,False,False,False,False,anual,True
8,S-24796e,A-cc8c8f,27/11/2024,Não fornecido,Enterprise,23,4577.0,54924.0,False,False,False,False,anual,False
9,S-d0c344,A-80eeb6,27/10/2024,Não fornecido,Pro,22,1078.0,12936.0,False,False,False,False,anual,False


### 3. ravenstack_feature_usage.csv
Uso diário por feature: contagem, duração, erros, flag de beta. Chave: `subscription_id`.

In [101]:
feature_usage = pd.read_csv(DATA_DIR / "ravenstack_feature_usage.csv")
feature_usage = feature_usage.rename(columns={
    "usage_id": "id_uso",
    "subscription_id": "id_assinatura",
    "usage_date": "data_uso",
    "feature_name": "nome_feature",
    "usage_count": "quantidade_usos",
    "usage_duration_secs": "duracao_uso_seg",
    "error_count": "contagem_erros",
    "is_beta_feature": "feature_beta",
})

# Data de uso em formato padrão Brasil (dd/mm/yyyy)
feature_usage["data_uso"] = pd.to_datetime(feature_usage["data_uso"], errors="coerce").dt.strftime("%d/%m/%Y")

# Duração: segundos -> minutos (nova coluna, remove segundos)
feature_usage["duracao_uso_min"] = (feature_usage["duracao_uso_seg"] / 60).round(2)
feature_usage = feature_usage.drop(columns=["duracao_uso_seg"])

# NaN em colunas de texto/categóricas -> "Não fornecido"
for col in feature_usage.select_dtypes(include="object").columns:
    feature_usage[col] = feature_usage[col].fillna("Não fornecido")

print("Shape:", feature_usage.shape)
print("\nColunas:", list(feature_usage.columns))
print("\nTipos:")
display(feature_usage.dtypes)
feature_usage.head(10)

Shape: (25000, 8)

Colunas: ['id_uso', 'id_assinatura', 'data_uso', 'nome_feature', 'quantidade_usos', 'contagem_erros', 'feature_beta', 'duracao_uso_min']

Tipos:


id_uso              object
id_assinatura       object
data_uso            object
nome_feature        object
quantidade_usos      int64
contagem_erros       int64
feature_beta          bool
duracao_uso_min    float64
dtype: object

,id_uso,id_assinatura,data_uso,nome_feature,quantidade_usos,contagem_erros,feature_beta,duracao_uso_min
0,U-1c6c24,S-0fcf7d,27/07/2023,feature_20,9,0,False,83.40
1,U-f07cb8,S-c25263,07/08/2023,feature_5,9,0,False,6.15
2,U-096807,S-f29e7f,07/12/2023,feature_3,9,0,False,24.30
3,U-6b1580,S-be655e,28/07/2024,feature_40,5,0,False,34.75
4,U-720a29,S-f9b1d0,02/12/2024,feature_12,12,0,False,15.00
5,U-c5692d,S-e958ba,31/10/2023,feature_14,11,0,False,68.38
6,U-7078db,S-0c9fdd,20/09/2023,feature_34,8,2,False,33.60
7,U-a60033,S-4e4ab7,05/06/2023,feature_27,10,0,False,91.17
8,U-5cd930,S-a3a1c1,20/02/2024,feature_18,9,1,False,84.45
9,U-ebb375,S-9e450b,09/02/2023,feature_28,6,0,False,46.30


Cruze **subscriptions** com **feature_usage** em `subscription_id` e adicione `account_id` em `feature_usage`.

In [102]:
# Merge: feature_usage + subscriptions (apenas id_conta)
feature_usage = feature_usage.merge(
    subscriptions[["id_assinatura", "id_conta"]],
    on="id_assinatura",
    how="left"
)
# NaN em id_conta (e demais object) após merge -> "Não fornecido"
for col in feature_usage.select_dtypes(include="object").columns:
    feature_usage[col] = feature_usage[col].fillna("Não fornecido")
print("Colunas após merge:", list(feature_usage.columns))
print("Shape:", feature_usage.shape)
feature_usage.head(10)

Colunas após merge: ['id_uso', 'id_assinatura', 'data_uso', 'nome_feature', 'quantidade_usos', 'contagem_erros', 'feature_beta', 'duracao_uso_min', 'id_conta']
Shape: (25000, 9)


,id_uso,id_assinatura,data_uso,nome_feature,quantidade_usos,contagem_erros,feature_beta,duracao_uso_min,id_conta
0,U-1c6c24,S-0fcf7d,27/07/2023,feature_20,9,0,False,83.40,A-e08cd3
1,U-f07cb8,S-c25263,07/08/2023,feature_5,9,0,False,6.15,A-c7ffc2
2,U-096807,S-f29e7f,07/12/2023,feature_3,9,0,False,24.30,A-bbe56f
3,U-6b1580,S-be655e,28/07/2024,feature_40,5,0,False,34.75,A-7f29a7
4,U-720a29,S-f9b1d0,02/12/2024,feature_12,12,0,False,15.00,A-65a46c
5,U-c5692d,S-e958ba,31/10/2023,feature_14,11,0,False,68.38,A-417d2f
6,U-7078db,S-0c9fdd,20/09/2023,feature_34,8,2,False,33.60,A-5b4ebb
7,U-a60033,S-4e4ab7,05/06/2023,feature_27,10,0,False,91.17,A-eb1312
8,U-5cd930,S-a3a1c1,20/02/2024,feature_18,9,1,False,84.45,A-f25509
9,U-ebb375,S-9e450b,09/02/2023,feature_28,6,0,False,46.30,A-89ee83


### 4. ravenstack_support_tickets.csv
Tickets: tempo de resolução, first response time, satisfação, escalações. Chave: `account_id`.

In [103]:
support_tickets = pd.read_csv(DATA_DIR / "ravenstack_support_tickets.csv")
support_tickets = support_tickets.rename(columns={
    "ticket_id": "id_ticket",
    "account_id": "id_conta",
    "submitted_at": "data_abertura",
    "closed_at": "data_fechamento",
    "resolution_time_hours": "tempo_resolucao_horas",
    "priority": "prioridade",
    "first_response_time_minutes": "tempo_primeira_resposta_min",
    "satisfaction_score": "nota_satisfacao",
    "escalation_flag": "flag_escalonamento",
})

# Datas em formato padrão Brasil (dd/mm/yyyy)
support_tickets["data_abertura"] = pd.to_datetime(support_tickets["data_abertura"], errors="coerce").dt.strftime("%d/%m/%Y")
# data_fechamento: apenas a data (dd/mm/yyyy)
support_tickets["data_fechamento"] = pd.to_datetime(support_tickets["data_fechamento"], errors="coerce").dt.strftime("%d/%m/%Y")

# NaN em colunas de texto/categóricas -> "Não fornecido"
for col in support_tickets.select_dtypes(include="object").columns:
    support_tickets[col] = support_tickets[col].fillna("Não fornecido")

print("Shape:", support_tickets.shape)
print("\nColunas:", list(support_tickets.columns))
print("\nTipos:")
display(support_tickets.dtypes)
support_tickets.head(10)

Shape: (2000, 9)

Colunas: ['id_ticket', 'id_conta', 'data_abertura', 'data_fechamento', 'tempo_resolucao_horas', 'prioridade', 'tempo_primeira_resposta_min', 'nota_satisfacao', 'flag_escalonamento']

Tipos:


id_ticket                       object
id_conta                        object
data_abertura                   object
data_fechamento                 object
tempo_resolucao_horas          float64
prioridade                      object
tempo_primeira_resposta_min      int64
nota_satisfacao                float64
flag_escalonamento                bool
dtype: object

,id_ticket,id_conta,data_abertura,data_fechamento,tempo_resolucao_horas,prioridade,tempo_primeira_resposta_min,nota_satisfacao,flag_escalonamento
0,T-0024de,A-712f1c,27/07/2023,28/07/2023,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,08/07/2024,09/07/2024,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,17/10/2024,17/10/2024,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,08/09/2024,09/09/2024,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,30/11/2024,01/12/2024,26.0,medium,8,NaN,False
5,T-90f06d,A-94c3cd,27/07/2023,27/07/2023,9.0,medium,60,NaN,False
6,T-30b537,A-2e4581,09/09/2023,10/09/2023,27.0,urgent,64,3.0,False
7,T-60242d,A-72799b,29/01/2024,29/01/2024,12.0,low,56,4.0,True
8,T-7119c9,A-b179bf,27/08/2023,27/08/2023,16.0,urgent,154,NaN,False
9,T-b0edf2,A-7cfe77,06/05/2024,07/05/2024,28.0,medium,150,3.0,False


### 5. ravenstack_churn_events.csv
Eventos de churn: reason code, valor de refund, feedback em texto. Chave: `account_id`.

In [104]:
churn_events = pd.read_csv(DATA_DIR / "ravenstack_churn_events.csv")
churn_events = churn_events.rename(columns={
    "churn_event_id": "id_evento_churn",
    "account_id": "id_conta",
    "churn_date": "data_churn",
    "reason_code": "motivo",
    "refund_amount_usd": "valor_reembolso_usd",
    "preceding_upgrade_flag": "flag_upgrade_anterior",
    "preceding_downgrade_flag": "flag_downgrade_anterior",
    "is_reactivation": "reativacao",
    "feedback_text": "feedback",
})

# Data em formato padrão Brasil (dd/mm/yyyy)
churn_events["data_churn"] = pd.to_datetime(churn_events["data_churn"], errors="coerce").dt.strftime("%d/%m/%Y")

# Padronização: motivo (código -> português)
churn_events["motivo"] = churn_events["motivo"].replace({
    "features": "Funcionalidades",
    "support": "Suporte",
    "budget": "Orçamento",
    "unknown": "Desconhecido",
    "competitor": "Concorrente",
    "pricing": "Preços",
})

# Padronização: feedback (texto -> português)
churn_events["feedback"] = churn_events["feedback"].replace({
    "switched to competitor": "Foi para a concorrência",
    "missing features": "Faltam funcionalidades",
    "too expensive": "Muito caro",
})

# NaN em colunas de texto/categóricas -> "Não fornecido"
for col in churn_events.select_dtypes(include="object").columns:
    churn_events[col] = churn_events[col].fillna("Não fornecido")

print("Shape:", churn_events.shape)
print("\nColunas:", list(churn_events.columns))
print("\nTipos:")
display(churn_events.dtypes)
churn_events.head(100)

Shape: (600, 9)

Colunas: ['id_evento_churn', 'id_conta', 'data_churn', 'motivo', 'valor_reembolso_usd', 'flag_upgrade_anterior', 'flag_downgrade_anterior', 'reativacao', 'feedback']

Tipos:


id_evento_churn             object
id_conta                    object
data_churn                  object
motivo                      object
valor_reembolso_usd        float64
flag_upgrade_anterior         bool
flag_downgrade_anterior       bool
reativacao                    bool
feedback                    object
dtype: object

,id_evento_churn,id_conta,data_churn,motivo,valor_reembolso_usd,flag_upgrade_anterior,flag_downgrade_anterior,reativacao,feedback
0,C-816288,A-c37cab,27/10/2024,Preços,4.03,False,False,False,Foi para a concorrência
1,C-5a81e7,A-37f969,25/06/2024,Suporte,96.45,True,False,False,Não fornecido
2,C-a174be,A-b07346,12/11/2024,Orçamento,0.00,False,False,False,Faltam funcionalidades
3,C-accb39,A-1e50e0,01/11/2023,Orçamento,54.94,False,False,False,Foi para a concorrência
4,C-92f889,A-956988,30/12/2024,Desconhecido,0.00,False,True,True,Muito caro
...,...,...,...,...,...,...,...,...,...
95,C-b8127e,A-d792a6,13/08/2024,Suporte,0.00,False,False,False,Faltam funcionalidades
96,C-8153b7,A-7cfe77,19/12/2024,Funcionalidades,0.00,False,True,False,Faltam funcionalidades
97,C-b6b828,A-69fad4,17/06/2024,Suporte,0.00,False,False,False,Não fornecido
98,C-a666a4,A-4bfa33,28/12/2024,Desconhecido,7.26,True,False,False,Foi para a concorrência


**Churn events — valores distintos em `motivo` e `feedback`**

In [105]:
print("=== MOTIVO (distintos) ===")
print(churn_events["motivo"].dropna().unique().tolist())
print("\nContagem por motivo:")
display(churn_events["motivo"].value_counts(dropna=False))

print("\n=== FEEDBACK (distintos) ===")
feedback_distintos = churn_events["feedback"].dropna().drop_duplicates()
print(f"Total de textos de feedback distintos: {len(feedback_distintos)}")
print("\nAmostra dos feedbacks (primeiros 20):")
for i, t in enumerate(feedback_distintos.head(20)):
    print(f"  {i+1}. {str(t)[:100]}{'...' if len(str(t)) > 100 else ''}")

=== MOTIVO (distintos) ===
['Preços', 'Suporte', 'Orçamento', 'Desconhecido', 'Funcionalidades', 'Concorrente']

Contagem por motivo:


motivo
Funcionalidades    114
Suporte            104
Orçamento          104
Desconhecido        95
Concorrente         92
Preços              91
Name: count, dtype: int64


=== FEEDBACK (distintos) ===
Total de textos de feedback distintos: 4

Amostra dos feedbacks (primeiros 20):
  1. Foi para a concorrência
  2. Não fornecido
  3. Faltam funcionalidades
  4. Muito caro


### Resumo dos 5 datasets

In [106]:
datasets = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events,
}
for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

accounts: 500 linhas × 10 colunas
subscriptions: 5,000 linhas × 14 colunas
feature_usage: 25,000 linhas × 9 colunas
support_tickets: 2,000 linhas × 9 colunas
churn_events: 600 linhas × 9 colunas
